# Data Ingestion Agent
**Section I — Data Understanding + Hypothesis Setup (Joey)**

**Responsibilities:**
- Load X5 RetailHero dataset (`clients`, `purchases`, `train`)
- Validate schema and check data quality
- Confirm treatment and target columns are present
- Check nulls, duplicates, join keys, row counts
- Split labeled customers into **train / val / test** (stratified on treatment × target)

> Feature engineering in later stages must **not** use target info from val/test customers.

**Outputs → `./artifacts/`**

In [ ]:
import pandas as pd
import numpy as np
from sklift.datasets import fetch_x5
from sklearn.model_selection import train_test_split
import os, warnings
warnings.filterwarnings('ignore')

OUTPUT_DIR = 'artifacts'
os.makedirs(OUTPUT_DIR, exist_ok=True)
RANDOM_STATE = 42

In [ ]:
print("Loading X5 RetailHero dataset...")
dataset   = fetch_x5()
data      = dataset.data
target    = dataset.target      # pd.Series - binary purchase indicator
treatment = dataset.treatment   # pd.Series - binary coupon flag

clients   = data.clients    # demographics for all known clients
train     = data.train      # labeled rows (client_id + experiment columns)
purchases = data.purchases  # full transaction history

print(f"\nRow counts:")
print(f"  clients   : {len(clients):>10,}")
print(f"  train     : {len(train):>10,}")
print(f"  purchases : {len(purchases):>10,}")
print(f"\nColumn names:")
print(f"  clients   : {list(clients.columns)}")
print(f"  train     : {list(train.columns)}")
print(f"  purchases : {list(purchases.columns)}")

## 1. Schema Validation

In [ ]:
REQUIRED = {
    "clients":   ["client_id"],
    "train":     ["client_id"],
    "purchases": ["client_id", "transaction_datetime", "purchase_sum"],
}
OPTIONAL = {
    "clients":   ["age", "gender"],
    "train":     ["first_issue_date", "first_redeem_date"],
    "purchases": ["transaction_id", "store_id", "product_id", "product_quantity"],
}

dfs = {"clients": clients, "train": train, "purchases": purchases}
issues = []
for name, req_cols in REQUIRED.items():
    df = dfs[name]
    missing = [c for c in req_cols if c not in df.columns]
    if missing:
        issues.append(f"[{name}] MISSING required cols: {missing}")
    else:
        opt_found = [c for c in OPTIONAL[name] if c in df.columns]
        print(f"\u2713 [{name}] required schema OK | optional present: {opt_found}")

if issues:
    for issue in issues:
        print(f"\u2717 {issue}")
    raise ValueError("Schema validation failed")
else:
    print("\n\u2713 All schema checks passed")

## 2. Treatment / Target Verification

In [ ]:
assert len(target) == len(treatment) == len(train), "Length mismatch between target/treatment/train"
assert set(target.unique()).issubset({0, 1}),    "target contains non-binary values"
assert set(treatment.unique()).issubset({0, 1}), "treatment contains non-binary values"

n_treated  = int(treatment.sum())
n_control  = int((treatment == 0).sum())
rate_t     = target[treatment == 1].mean()
rate_c     = target[treatment == 0].mean()
ate        = rate_t - rate_c

print("Treatment / Target distribution:")
print(f"  Treated  : {n_treated:,} ({n_treated/len(treatment):.1%})")
print(f"  Control  : {n_control:,} ({n_control/len(treatment):.1%})")
print(f"  Target rate (treated) : {rate_t:.4f}")
print(f"  Target rate (control) : {rate_c:.4f}")
print(f"  Unconditional ATE     : {ate:+.4f}")
print("\n\u2713 Treatment/target alignment verified")

## 3. Null Checks

In [ ]:
def null_report(df, name):
    null_pct = df.isnull().mean()
    null_pct = null_pct[null_pct > 0]
    if null_pct.empty:
        print(f"\u2713 [{name}] No nulls")
    else:
        print(f"\u2139 [{name}] Null rates:")
        for col, pct in null_pct.items():
            flag = "  \u26a0 HIGH" if pct > 0.30 else ""
            print(f"    {col:<30} {pct:.2%}{flag}")

null_report(clients, "clients")
null_report(train, "train")
null_report(
    purchases.sample(min(500_000, len(purchases)), random_state=RANDOM_STATE),
    "purchases (500k sample)"
)

## 4. Duplicate & Join Key Checks

In [ ]:
def dup_check(df, name, key):
    n = df.duplicated(subset=[key]).sum()
    symbol = "\u2713" if n == 0 else "\u26a0"
    print(f"{symbol} [{name}] Duplicate '{key}': {n:,}")

dup_check(clients, "clients", "client_id")
dup_check(train,   "train",   "client_id")
if "transaction_id" in purchases.columns:
    dup_check(purchases, "purchases", "transaction_id")

pct_train_in_clients = train["client_id"].isin(clients["client_id"]).mean()
pct_purch_in_clients = purchases["client_id"].isin(clients["client_id"]).mean()
print(f"\n\u2139 Train clients found in clients table : {pct_train_in_clients:.2%}")
print(f"\u2139 Purchase clients found in clients    : {pct_purch_in_clients:.2%}")

## 5. Date Parsing & Summary

In [ ]:
if "transaction_datetime" in purchases.columns:
    purchases["transaction_datetime"] = pd.to_datetime(purchases["transaction_datetime"])
    print(f"Purchase dates  : {purchases['transaction_datetime'].min().date()} to {purchases['transaction_datetime'].max().date()}")

if "first_issue_date" in train.columns:
    train["first_issue_date"] = pd.to_datetime(train["first_issue_date"])
    print(f"Treatment dates : {train['first_issue_date'].min().date()} to {train['first_issue_date'].max().date()}")

if "first_redeem_date" in train.columns:
    train["first_redeem_date"] = pd.to_datetime(train["first_redeem_date"])
    redeem_pct = train["first_redeem_date"].notna().mean()
    print(f"Coupon redemption rate : {redeem_pct:.2%}  (post-treatment -- will NOT be used as a feature)")

print(f"\nRow counts confirmed:")
print(f"  clients   : {len(clients):,}")
print(f"  train     : {len(train):,}")
print(f"  purchases : {len(purchases):,}")

## 6. Train / Validation / Test Split

**Strategy: 70 / 15 / 15 stratified on `treatment x target`**

Stratification on the 2x2 cross of treatment and target ensures each split preserves
both the treatment assignment rate and the base purchase rate.
This is essential for uplift modeling where both rates must be consistent across folds.

In [ ]:
strat_labels = treatment.astype(str) + "_" + target.astype(str)

idx_trainval, idx_test = train_test_split(
    train.index,
    test_size=0.15,
    stratify=strat_labels,
    random_state=RANDOM_STATE,
)
idx_train, idx_val = train_test_split(
    idx_trainval,
    test_size=0.15 / 0.85,   # ~17.6% of trainval => 15% of total
    stratify=strat_labels.loc[idx_trainval],
    random_state=RANDOM_STATE,
)

print("Split sizes:")
print(f"  train : {len(idx_train):,}  ({len(idx_train)/len(train):.1%})")
print(f"  val   : {len(idx_val):,}  ({len(idx_val)/len(train):.1%})")
print(f"  test  : {len(idx_test):,}  ({len(idx_test)/len(train):.1%})")
total_check = len(idx_train) + len(idx_val) + len(idx_test)
print(f"  total : {total_check:,} (expected {len(train):,})")

print("\nDistribution check (should be consistent across splits):")
print(f"  {'split':<8} {'treatment rate':>14} {'target rate':>11}")
for split_name, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test)]:
    t = treatment.loc[idx]
    y = target.loc[idx]
    print(f"  {split_name:<8} {t.mean():>14.4f} {y.mean():>11.4f}")

## 7. Save Artifacts

In [ ]:
def build_split_df(base_df, idx, target_s, treatment_s):
    df = base_df.loc[idx].copy().reset_index(drop=True)
    df["target"]    = target_s.loc[idx].values
    df["treatment"] = treatment_s.loc[idx].values
    return df

train_save = build_split_df(train, idx_train, target, treatment)
val_save   = build_split_df(train, idx_val,   target, treatment)
test_save  = build_split_df(train, idx_test,  target, treatment)

train_save.to_parquet(f"{OUTPUT_DIR}/train_split.parquet",  index=False)
val_save.to_parquet(  f"{OUTPUT_DIR}/val_split.parquet",    index=False)
test_save.to_parquet( f"{OUTPUT_DIR}/test_split.parquet",   index=False)
clients.to_parquet(   f"{OUTPUT_DIR}/clients.parquet",      index=False)
purchases.to_parquet( f"{OUTPUT_DIR}/purchases.parquet",    index=False)

print("\u2713 Artifacts saved to ./artifacts/:")
for fname, df in [
    ("train_split.parquet",  train_save),
    ("val_split.parquet",    val_save),
    ("test_split.parquet",   test_save),
    ("clients.parquet",      clients),
    ("purchases.parquet",    purchases),
]:
    print(f"  {fname:<35} {len(df):>10,} rows x {df.shape[1]} cols")

In [ ]:
print("=" * 55)
print("DATA INGESTION AGENT - SUMMARY REPORT")
print("=" * 55)
print(f"Dataset            : X5 RetailHero")
print(f"Labeled clients    : {len(train):,}")
print(f"  Treated          : {n_treated:,}  ({n_treated/len(treatment):.1%})")
print(f"  Control          : {n_control:,}  ({n_control/len(treatment):.1%})")
print(f"Unconditional ATE  : {ate:+.4f}")
print(f"Purchase records   : {len(purchases):,}")
print(f"Unique buyers      : {purchases['client_id'].nunique():,}")
print(f"\nSplit (70/15/15 stratified on treatment x target):")
print(f"  train : {len(idx_train):,}")
print(f"  val   : {len(idx_val):,}")
print(f"  test  : {len(idx_test):,}")
print(f"\nAll artifacts saved to ./artifacts/")
print("=" * 55)